# Saturation level of each channel over time

*Draft prepared with Claude Code for A. Villano, 2026-11-02.*

Does the pulse height at which a channel saturates change during Run 76? For each of the six channels of the Side-2 readout, the **pulse height** (`current_max` of `goldsluice`'s `peak`: the largest sample above the baseline, in ADC counts) is drawn dump by dump along the run, in a long horizontal plot, one row per channel. A saturated channel stops at its own ceiling, so the **upper envelope** of the heights is the saturation level; a level that moves is a change of the gain or the bias of the channel.

**The data.** The eight DCRC1pB-triggered series near -82 V (-82.2 to -81.6 V; trigger `DCRC1pB 10`, 436 dumps), processed by `r76-82v-dcrc1pb.dataset.toml` (output in `~/goldsluice-output/r76-82v-dcrc1pb`). The voltage and trigger of every series are from `series.csv` of the processing folder. The series are in order of their start time (the eight digits after `0722` of a series name are month, day, hour and minute of 2022).

**The time axis.** One dump is about one minute of data (the duration of a series in minutes is its number of dumps), so each dump is placed at `series start + (dump - 1)` minutes. The weeks between the series are **left out**: the series follow each other and are separated by a vertical line, labelled with the date, the high voltage and the trigger. Real triggers only (`Unknown`; see `integral-on-the-face.ipynb`).

Run with the `darkmatter_cli_env` kernel.

In [ ]:
import csv
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from goldpan.dataset import Dataset

LETTERS = "ABCDEF"

def field(tab, k, name):
    # one quantity of one channel: the channel is a struct column of the Parquet file
    return pc.struct_field(tab[f"Side2p{k}"].combine_chunks(), name).to_numpy(zero_copy_only=False)

_here = Path.cwd().resolve()
PROCESSING = next(p / "data_analysis" / "R76" / "processing" for p in [_here, *_here.parents] if (p / "data_analysis" / "R76" / "processing").is_dir())
DEFINITIONS = ["r76-82v-dcrc1pb.dataset.toml"]
OUTPUTS = [Path(Dataset.open(PROCESSING / d).spec.output) for d in DEFINITIONS]
CACHE = Path.home() / "goldsluice-output" / "relcal-height-per-dump-82v.npz"

with open(PROCESSING / "series.csv", newline="") as fh:
    meta = {r["series"]: r for r in csv.DictReader(fh)}

## Heights per dump

For every dump and channel: the 50th, 99th and 99.9th percentile of the height of the real triggers, and their maximum. The percentiles are used rather than the maximum alone because a few events have a glitch of several thousand counts (see `integral-vs-height.ipynb`). Reading the 436 files takes a minute or two the first time; the result is cached in `~/goldsluice-output/relcal-height-per-dump-82v.npz`.

In [ ]:
QS = [50, 99, 99.9]
if CACHE.exists():
    z = np.load(CACHE, allow_pickle=False)
    series_d, dump_d, nev, stat = z["series"], z["dump"], z["nev"], z["stat"]
else:
    S_, D_, N_, ST_ = [], [], [], []
    cols = ["trigger_type"] + [f"Side2p{k}" for k in LETTERS]
    for out in OUTPUTS:
        for f in sorted((out / "NAI-S2").glob("*_F[0-9][0-9][0-9][0-9].parquet")):
            tab = pq.read_table(f, columns=cols)
            real = np.asarray(tab["trigger_type"].to_pylist()) == "Unknown"
            h = np.column_stack([field(tab, k, "current_max") for k in LETTERS])[real]
            s, d = f.stem.rsplit("_F", 1)
            S_.append(s); D_.append(int(d)); N_.append(len(h))
            ST_.append(np.full((6, 4), np.nan) if len(h) < 20 else
                       np.column_stack([np.percentile(h, QS, axis=0).T, h.max(axis=0)]))   # (channel, [p50, p99, p99.9, max])
    series_d, dump_d, nev, stat = np.array(S_), np.array(D_), np.array(N_), np.array(ST_)
    np.savez(CACHE, series=series_d, dump=dump_d, nev=nev, stat=stat)
order = np.lexsort((dump_d, series_d))
series_d, dump_d, nev, stat = series_d[order], dump_d[order], nev[order], stat[order]
print(f"{len(dump_d):,} dumps in {len(np.unique(series_d))} series; real triggers per dump: median {int(np.median(nev))}, "
      f"{(nev < 20).sum()} dumps with fewer than 20 (left blank)")

## The run, channel by channel

Per dump: the 99th percentile of the height (line) and the maximum (faint dots). The vertical scale ends just above the plateau, so the few glitches of several thousand counts (the faint dots) are cut off. The series are shaded alternately; the labels give the start (month-day hour:minute of 2022), the high voltage and the trigger.

In [ ]:
x = np.zeros(len(dump_d))              # position along the plot: the dumps in order, in minutes, series one after the other
bounds, starts = [], []
pos = 0.0
for s in np.unique(series_d):
    m = series_d == s
    x[m] = pos + dump_d[m] - 1
    bounds.append((pos, pos + dump_d[m].max()))
    starts.append(s)
    pos += dump_d[m].max() + 20           # a gap of 20 minutes between series, for the separating line

def label(s):
    when = datetime(2022, int(s[4:6]), int(s[6:8]), int(s[9:11]), int(s[11:13]))
    r = meta.get(s, {})
    hv = f"{float(r['hv_v']):.1f} V" if r else "?"
    return f"{when:%m-%d %H:%M}\n{hv}\n{r.get('trigger', '?')}"

fig, axes = plt.subplots(6, 1, figsize=(26, 17), sharex=True)
for i, (ax, k) in enumerate(zip(axes, LETTERS)):
    for j, (a, b) in enumerate(bounds):
        if j % 2 == 0:
            ax.axvspan(a - 10, b + 10, color="0.93", zorder=0)
    ax.plot(x, stat[:, i, 3], ".", ms=1.5, color="0.55", zorder=2)
    for s_ in starts:                                              # one line per series: none across the gaps between them
        m_ = series_d == s_
        ax.plot(x[m_], stat[m_, i, 1], "-", lw=0.8, color="C0", zorder=3)
    ax.set_ylabel(f"Side2p{k}\nheight (counts)")
    ax.set_ylim(0, 1.3 * np.nanmax(stat[:, i, 1]))        # the plateau; the glitches of several thousand counts above it are cut off
    ax.set_xlim(-30, pos)
for (a, b), s in zip(bounds, starts):
    axes[0].text((a + b) / 2, 1.02, label(s), transform=axes[0].get_xaxis_transform(), ha="center", va="bottom", fontsize=7)
axes[-1].set_xlabel("time along the run (minutes of data; the gaps between series are left out)")
axes[-1].set_xticks([])
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

## The saturation level by series

The 99.99th percentile of the height of the real triggers of each series (the same definition as in `integral-vs-height.ipynb`, there over the whole run), channel by channel, and the same relative to the median over the series. A saturated channel is flat at its ceiling, so a change of this number from one series to the next is a change of the level.

In [ ]:
# the 99.99th percentile needs the events, not the per-dump statistics: read them series by series
LEVEL_CACHE = Path.home() / "goldsluice-output" / "relcal-saturation-per-series-82v.npz"
sers = list(np.unique(series_d))
if LEVEL_CACHE.exists():
    level = np.load(LEVEL_CACHE)["level"]
else:
    level = np.full((len(sers), 6), np.nan)
    cols = ["trigger_type"] + [f"Side2p{k}" for k in LETTERS]
    for n, s in enumerate(sers):
        parts = []
        for out in OUTPUTS:
            files = sorted((out / "NAI-S2").glob(f"{s}_F[0-9][0-9][0-9][0-9].parquet"))
            if files:
                parts = [pq.read_table(f, columns=cols) for f in files]
                break
        tab = pa.concat_tables(parts)
        real = np.asarray(tab["trigger_type"].to_pylist()) == "Unknown"
        h = np.column_stack([field(tab, k, "current_max") for k in LETTERS])[real]
        level[n] = np.percentile(h, 99.99, axis=0)
    np.savez(LEVEL_CACHE, level=level)

print(f"{'series':>14} {'HV (V)':>7} {'trigger':>12}" + "".join(f"{'Side2p' + k:>10}" for k in LETTERS))
for s, row in zip(sers, level):
    r = meta[s]
    print(f"{s:>14} {float(r['hv_v']):>7.1f} {r['trigger']:>12}" + "".join(f"{v:>10.0f}" for v in row))
print(f"{'median':>35}" + "".join(f"{v:>10.0f}" for v in np.median(level, axis=0)))
print(f"{'largest change from median':>35}" + "".join(f"{np.max(np.abs(level[:, i] / np.median(level[:, i]) - 1)):>10.1%}" for i in range(6)))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
hv = np.array([float(meta[s]["hv_v"]) for s in sers])
for i, k in enumerate(LETTERS):
    ax1.plot(range(len(sers)), level[:, i] / np.median(level[:, i]), "o-", label=f"Side2p{k}")
    ax2.plot(hv, level[:, i] / np.median(level[:, i]), "o", label=f"Side2p{k}")
ax1.set_xticks(range(len(sers)))
ax1.set_xticklabels([s[4:] for s in sers], rotation=90, fontsize=7)
ax1.set_xlabel("series, in order of time (month day _ hour minute)")
ax2.set_xlabel("high voltage (V)")
for ax in (ax1, ax2):
    ax.set_ylabel("saturated height / its median over the series")
    ax.axhline(1, color="0.5", lw=0.5)
ax1.legend(ncol=3, fontsize=8)
fig.tight_layout()
plt.show()

**What this shows, and what is not done.** The height ceiling of every channel dump by dump over the eight -82 V series, and per series. Not done: the seven NaI-triggered series near -82 V (not processed yet) and an explanation of any step; a change in the level at the start of a series may be a change of the bias of the channel (the high voltage changes slightly between the series, -82.2 to -81.6 V) or of the readout, which this plot cannot tell apart. The ceiling of the integral is this height times the 2.88 ms that is left of the trace after the pulse starts (`spectrum-82v-total.ipynb`).